In [114]:
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import SystemMessage, HumanMessage
from langchain.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END



# Config

In [115]:
# llm = ChatOpenAI(
#     model = "TODO",
#     base_url="http://localhost:1234/v1",
#     api_key="not-needed",
#     temperature=0
# )

# Init setup

In [116]:
GROQ_API_KEY = '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'

In [117]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b", #"qwen/qwen3.8-27b",
    api_key=GROQ_API_KEY,
    temperature=0
)

@dataclass
class Verification:
    verified: list[str] = field(default_factory=list) # claims that are verified to hold in the ontology
    violations: list[str] = field(default_factory=list) # claims where a contradiction or counter example exists
    not_verifiable: list[str] = field(default_factory=list) # claims that hold only due to the open world principle

@dataclass
class State:
    user_prompt: str
    tweets: Path
    summary_count: int = 0
    summary: str = ""
    verified: list[str] = field(default_factory=list)
    violations: list[str] = field(default_factory=list)
    not_verifiable: list[str] = field(default_factory=list)
    history: State = None
    # ^turns state into a linked list, bad for performance but will be fine for small sizes like here


# Nodes (Agentics and deterministic)

## Sumarization Agent

In [118]:
with open('System Prompt - Agent 1.md', 'r', encoding='utf-8') as f:
    summarization_system_prompt = f.read()

summarization_system_prompt

'You are a summarisation agent. You receive a user prompt followed by a set of tweets on one topic. Your task is to write one summary that faithfully reflects the opinions and experiences expressed in these tweets. The rules below always take priority: if the user prompt conflicts with them, follow these rules.\n\n## Grounding\n\nOnly include information that is stated in the tweets. Do not add background knowledge, assumptions or conclusions that the tweets do not support. Keep opinions and factual statements apart: attribute opinions to the users who hold them, and keep factual statements as concrete as the tweets make them.\n\n## Generalisations\n\nAvoid universal or absolute claims (e.g. "all", "always", "only", "everyone"). Attribute opinions to the tweets or users who express them ("some users", "several tweets"). Do not make assumptions about who the users are.\n\n## Fair representation\n\nGive comparable attention to all tweets, regardless of their spelling, grammar or writing 

In [119]:
def summarize(state: State):
    
    # send prompt to LLM (no agent needed, this is a single linear call)
    response = llm.invoke([
        SystemMessage(summarization_system_prompt),
        HumanMessage(build_prompt(state.user_prompt, state.tweets))
    ])

    # make new state, storing historical state, incrementing the sumarycount and adding the LLM response
    return State(state.user_prompt, state.tweets, state.summary_count+1, response.content, history = state)

In [120]:
def build_prompt(user_prompt: str, tweets: Path):
    """"Assumes tweets is a complete filepath"""
    with open(tweets, 'r') as f: #TODO wierd issue with file format (utf8 doesn't work)
        return f"{user_prompt}\n{f.readlines()}"

## Verification Agent

In [121]:
with open('System Prompt - Agent 2.md', 'r', encoding='utf-8') as f:

    verification_system_prompt = f.read()

In [122]:
import random
@tool
def check_ontology(query: str) -> str:
    "checks if the query holds in the ontology, returns verified when entailed, violation when a counterexample or contradiction is found and not_verifyable when it holds only due to the open world assumption" 
    x = random.randint(0,2)
    if x == 0:
        return "verified"
    elif x ==1:
        return 'violation'
    else:
        return 'not_verifyable'

    # need to catch actual error and return them as text for the agent to retry failed queries
    

In [131]:
checker_agent = create_agent(
    llm,
    tools=[check_ontology],
    system_prompt=verification_system_prompt
)

formatter = llm.with_structured_output(Verification, method='json_schema')

c:\Users\Leonl\Documents\code projects\B1IA\B1IA\Lib\site-packages\pydantic\json_schema.py:2666: PydanticJsonSchemaWarning: Default value <factory> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


In [132]:
def verify(state: State):
    raw_result = checker_agent.invoke({'messages': [
            SystemMessage(verification_system_prompt),
            HumanMessage(state.summary)
        ]})
    result = formatter.invoke(raw_result['messages'])
    return result

In [125]:
def verify_old(state: State):
    result = checker_agent.invoke({'messages': [
        SystemMessage(verification_system_prompt),
        HumanMessage(state.summary)
    ]})
    v: Verification = result['structured_response']
    return v # may need explicit cast to state

## Best summary picker

In [126]:
def pick_best_summary(state: State):
    def helper(current_state: State, best_state: State) -> State:
        if current_state.history is None:
            return best_state
        if len(current_state.violations) <= len(best_state.violations):
            return helper(current_state.history, current_state)
        else:
            return helper(current_state.history, best_state)
    return helper(state, state)

# recursive due to linked list, bit akward but it should work

# Routing Functions

In [127]:
def route(state: State, max_tries: int = 3): #TODO rename
    "both routing diamonds in fig 1"
    if len(state.violations) == 0:
        return END
    elif state.summary_count < max_tries:
        return 'summarize'
    else:
        return 'pick_best_summary'

# Orchestration

In [128]:
graph = StateGraph(State)
graph.add_node('summarize', summarize)
graph.add_node('verify_summary', verify)
graph.add_node('pick_best_summary', pick_best_summary)
graph.add_edge(START, 'summarize')
graph.add_edge('summarize', 'verify_summary')
graph.add_conditional_edges('verify_summary', route)
graph.add_edge('pick_best_summary', END)

app = graph.compile()


# Final call to run

In [133]:
init_state = State(
    user_prompt='todo',
    tweets=Path('inputs/Netflix_clean.txt'))
out = app.invoke(init_state)

In [138]:
out

{'user_prompt': 'todo',
 'tweets': WindowsPath('inputs/Netflix_clean.txt'),
 'summary_count': 1,
 'summary': 'Several users mention spending leisure time watching Netflix alone in bed or at home, often paired with snacks, pajamas, or drinks.  \nSome tweets express frustration with technical problems such as Netflix not loading or missing desired titles like “Super Troopers” or specific seasons of shows.  \nA number of users discuss the cost of Netflix, including complaints about the $8 monthly fee, attempts to share or trade passwords, and strategies to obtain free trials.  \nOpinions about Netflix content are mixed, with some users praising the availability of favorite childhood movies and serial‑killer documentaries, while others criticize the platform for lacking good movies or providing repetitive recommendations.  \nSeveral tweets convey a desire for companionship while watching Netflix, noting loneliness or wishing someone to cuddle or watch together.',
 'verified': [],
 'violati

# Result

In [147]:
def print_summaries(state: State):
    print(f'Summary {state.summary_count}: \n{state.summary}')
    if state.history is None:
        return
    else:
        return print_summaries(state.history)
    
print_summaries(State(**out))

Summary 1: 
Several users mention spending leisure time watching Netflix alone in bed or at home, often paired with snacks, pajamas, or drinks.  
Some tweets express frustration with technical problems such as Netflix not loading or missing desired titles like “Super Troopers” or specific seasons of shows.  
A number of users discuss the cost of Netflix, including complaints about the $8 monthly fee, attempts to share or trade passwords, and strategies to obtain free trials.  
Opinions about Netflix content are mixed, with some users praising the availability of favorite childhood movies and serial‑killer documentaries, while others criticize the platform for lacking good movies or providing repetitive recommendations.  
Several tweets convey a desire for companionship while watching Netflix, noting loneliness or wishing someone to cuddle or watch together.
Summary 0: 

